# 🏋️ RepSense AI — WhatsApp Backend (Colab)
**Run each cell top-to-bottom. ~3 minutes to a live webhook URL.**

---
### Before running: add these 5 Secrets
Click the 🔑 **key icon** in the left sidebar → **+ Add new secret** for each:

| Secret name | Where to get it |
|---|---|
| `ANTHROPIC_API_KEY` | https://console.anthropic.com/settings/api-keys |
| `NGROK_AUTH_TOKEN` | https://dashboard.ngrok.com/get-started/your-authtoken (free signup) |
| `WHATSAPP_PHONE_NUMBER_ID` | Meta Dashboard → your app → WhatsApp → API Setup |
| `WHATSAPP_ACCESS_TOKEN` | Same page — the temporary token |
| `WHATSAPP_VERIFY_TOKEN` | Invent any string, e.g. `repsense_secret_2024` |

> Toggle **"Notebook access"** ON for every secret!

In [ ]:
# ── CELL 1: Install + create package folder ──────────────────────────────
!pip install fastapi "uvicorn[standard]" httpx anthropic pyngrok nest-asyncio apscheduler -q
!mkdir -p repsense
print('✅ Packages installed, repsense/ folder ready')

In [ ]:
# ── CELL 2: Load secrets ─────────────────────────────────────────────────
import os
from google.colab import userdata

def get_secret(key, fallback=''):
    try:
        val = userdata.get(key)
        if val:
            return val
    except Exception:
        pass
    print(f'❌  Secret "{key}" not found — add it in the 🔑 Secrets panel then re-run this cell')
    return fallback

os.environ['ANTHROPIC_API_KEY']        = get_secret('ANTHROPIC_API_KEY')
os.environ['WHATSAPP_PHONE_NUMBER_ID'] = get_secret('WHATSAPP_PHONE_NUMBER_ID')
os.environ['WHATSAPP_ACCESS_TOKEN']    = get_secret('WHATSAPP_ACCESS_TOKEN')
os.environ['WHATSAPP_VERIFY_TOKEN']    = get_secret('WHATSAPP_VERIFY_TOKEN', 'repsense_secret_2024')
NGROK_AUTH_TOKEN                       = get_secret('NGROK_AUTH_TOKEN')

print()
for k in ['ANTHROPIC_API_KEY', 'WHATSAPP_PHONE_NUMBER_ID', 'WHATSAPP_ACCESS_TOKEN', 'WHATSAPP_VERIFY_TOKEN']:
    v = os.environ.get(k, '')
    print(f'  {"✅" if v else "❌"}  {k}')
print(f'  {"✅" if NGROK_AUTH_TOKEN else "❌"}  NGROK_AUTH_TOKEN')
print()
print('Verify token:', os.environ.get('WHATSAPP_VERIFY_TOKEN'))

In [ ]:
%%writefile repsense/__init__.py
# RepSense AI package

In [ ]:
%%writefile repsense/whatsapp_client.py
import os
import httpx


class WhatsAppClient:
    BASE_URL = "https://graph.facebook.com/v19.0"

    def __init__(self):
        self.phone_id = os.getenv("WHATSAPP_PHONE_NUMBER_ID")
        self.token    = os.getenv("WHATSAPP_ACCESS_TOKEN")

    @property
    def _headers(self):
        return {"Authorization": f"Bearer {self.token}", "Content-Type": "application/json"}

    @property
    def _url(self):
        return f"{self.BASE_URL}/{self.phone_id}/messages"

    async def send_text(self, to: str, body: str):
        payload = {
            "messaging_product": "whatsapp",
            "recipient_type": "individual",
            "to": to,
            "type": "text",
            "text": {"preview_url": False, "body": body},
        }
        if not self.phone_id or not self.token:
            print(f"[MOCK] To {to}: {body[:100]}")
            return
        async with httpx.AsyncClient(timeout=10) as client:
            r = await client.post(self._url, json=payload, headers=self._headers)
        print(f"WhatsApp sent to {to}: HTTP {r.status_code}")
        return r.json()

    async def send_reaction(self, to: str, message_id: str, emoji: str):
        payload = {
            "messaging_product": "whatsapp",
            "to": to,
            "type": "reaction",
            "reaction": {"message_id": message_id, "emoji": emoji},
        }
        if not self.phone_id or not self.token:
            return
        async with httpx.AsyncClient(timeout=10) as client:
            await client.post(self._url, json=payload, headers=self._headers)

In [ ]:
%%writefile repsense/store.py
import json
import os
from datetime import datetime, timedelta
from threading import Lock

DATA_FILE = "user_data.json"


class UserStore:
    def __init__(self):
        self._lock = Lock()
        self._data = {}
        self._load()

    def get_or_create(self, phone: str) -> dict:
        with self._lock:
            if phone not in self._data:
                self._data[phone] = {
                    "phone": phone,
                    "name": "",
                    "weight_kg": None,
                    "goal": "",
                    "diet": "",
                    "sessions_this_week": 0,
                    "total_sessions": 0,
                    "last_session_date": None,
                    "reminders_on": True,
                    "joined": datetime.now().strftime("%Y-%m-%d"),
                }
            return self._data[phone]

    def save(self, phone: str, user: dict):
        with self._lock:
            self._data[phone] = user
            self._persist()

    def all_users(self) -> dict:
        with self._lock:
            return dict(self._data)

    def _load(self):
        if os.path.exists(DATA_FILE):
            try:
                with open(DATA_FILE) as f:
                    self._data = json.load(f)
            except Exception:
                self._data = {}

    def _persist(self):
        try:
            with open(DATA_FILE, "w") as f:
                json.dump(self._data, f, indent=2, default=str)
        except Exception as e:
            print(f"Store save error: {e}")

In [ ]:
%%writefile repsense/ai_coach.py
import os
import json
import re
from anthropic import AsyncAnthropic
from datetime import datetime


SYSTEM_PROMPT = """
You are RepSense AI — an intelligent fitness coach and nutrition advisor on WhatsApp.
Personality: encouraging, concise, knowledgeable — like a friendly personal trainer.

You help with:
1. Workout logging — parse what they trained, give ONE specific form tip, track session
2. Nutrition — personalised meal plans respecting cultural/dietary preferences
3. Supplements — evidence-based guidance (always recommend consulting a doctor)
4. Progress — weekly session counts and motivational insights
5. Profile setup — collect name, weight, goal, diet type on first message or /setup

FORMATTING (WhatsApp markdown only):
- *bold* for key numbers and exercise names
- _italic_ for tips and insights
- Bullet points with • (never dashes)
- Under 300 words per reply — this is a chat, not an essay
- Maximum 2 emojis per message

SPECIAL COMMANDS: /setup, /stats, /nutrition, /supplements, /reminders on|off, /reset

When a profile field changes OR a workout is logged, append this block at the END of your reply:
<<<JSON>>>
{"profile_updates": {}, "session_logged": false, "sessions_increment": 0}
<<<END>>>

Only include fields that actually changed in profile_updates.
Set session_logged: true and sessions_increment: 1 when a workout is logged.
If nothing changed, omit the block entirely.
"""


class AICoach:
    def __init__(self):
        self.client = AsyncAnthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
        self.model  = os.getenv("CLAUDE_MODEL", "claude-haiku-4-5-20251001")

    async def handle_message(self, user: dict, message: str) -> str:
        lower = message.lower().strip()

        # Fast-path local commands
        if lower in ("/help", "help", "hi", "hello", "hey", "start"):
            return self._welcome(user)
        if lower == "/stats":
            return self._stats(user)
        if lower.startswith("/reminders"):
            return self._reminders(user, lower)
        if lower == "/reset":
            user.clear()
            return "Profile reset! Send *hello* to start fresh."

        # Build profile context for Claude
        name    = user.get("name") or "Unknown"
        weight  = user.get("weight_kg") or "?"
        goal    = user.get("goal") or "not set"
        diet    = user.get("diet") or "not set"
        weekly  = user.get("sessions_this_week", 0)
        total   = user.get("total_sessions", 0)

        profile_ctx = (
            "User profile:\n"
            f"Name: {name} | Weight: {weight}kg\n"
            f"Goal: {goal} | Diet: {diet}\n"
            f"Sessions this week: {weekly} | Total sessions: {total}"
        )

        try:
            resp = await self.client.messages.create(
                model=self.model,
                max_tokens=600,
                system=SYSTEM_PROMPT,
                messages=[{"role": "user", "content": profile_ctx + "\n\nMessage: " + message}],
            )
            raw = resp.content[0].text
        except Exception as e:
            print(f"Claude error: {e}")
            return "Sorry, having a moment! Try again in a few seconds."

        reply, updates = self._parse_updates(raw)
        if updates:
            for k, v in updates.get("profile_updates", {}).items():
                if v not in (None, "", 0):
                    user[k] = v
            if updates.get("session_logged"):
                inc = updates.get("sessions_increment", 1)
                user["sessions_this_week"] = user.get("sessions_this_week", 0) + inc
                user["total_sessions"]     = user.get("total_sessions", 0) + inc
                user["last_session_date"]  = datetime.now().strftime("%Y-%m-%d")
        return reply.strip()

    def _parse_updates(self, raw: str):
        pattern = r"<<<JSON>>>(.*?)<<<END>>>"
        m = re.search(pattern, raw, re.DOTALL)
        if not m:
            return raw, {}
        try:
            return raw[:m.start()].strip(), json.loads(m.group(1).strip())
        except json.JSONDecodeError:
            return raw[:m.start()].strip(), {}

    def _welcome(self, user: dict) -> str:
        name = user.get("name", "")
        greeting = ("Hey " + name + "! 👋") if name else "Hey! 👋 Welcome to RepSense AI"
        return (
            greeting + "\n\n"
            "Here's what I can do:\n\n"
            "• 💪 Log workouts — _'squats 4x10 at 80kg'_\n"
            "• 🥗 Nutrition plan — /nutrition\n"
            "• 📊 Your stats — /stats\n"
            "• 💊 Supplements — /supplements\n"
            "• ⚙️ Set up profile — /setup\n\n"
            "What did you train today? 🏋️"
        )

    def _stats(self, user: dict) -> str:
        s    = user.get("sessions_this_week", 0)
        t    = user.get("total_sessions", 0)
        goal = user.get("goal") or "not set"
        msg  = "Great consistency! Keep it up 🔥" if s >= 3 else "Still time to hit your weekly target! 💪"
        return (
            "📊 *Your stats:*\n\n"
            f"• Sessions this week: *{s}*\n"
            f"• Total sessions: *{t}*\n"
            f"• Goal: *{goal}*\n\n"
            + msg
        )

    def _reminders(self, user: dict, cmd: str) -> str:
        if "off" in cmd:
            user["reminders_on"] = False
            return "Reminders turned *off*. Send /reminders on to re-enable."
        user["reminders_on"] = True
        return "Reminders turned *on*! You'll hear from me every morning at 7am. 💪"

In [ ]:
%%writefile repsense/app.py
import os
from fastapi import FastAPI, Request, Response, BackgroundTasks
from fastapi.responses import PlainTextResponse

from repsense.whatsapp_client import WhatsAppClient
from repsense.ai_coach import AICoach
from repsense.store import UserStore

app   = FastAPI(title="RepSense AI")
wa    = WhatsAppClient()
coach = AICoach()
store = UserStore()

VERIFY_TOKEN = os.getenv("WHATSAPP_VERIFY_TOKEN", "repsense_secret_2024")


@app.get("/")
def root():
    return {"status": "RepSense AI is running"}


@app.get("/webhook")
async def verify(request: Request):
    p = dict(request.query_params)
    if p.get("hub.mode") == "subscribe" and p.get("hub.verify_token") == VERIFY_TOKEN:
        print("Webhook verified!")
        return PlainTextResponse(p["hub.challenge"])
    print(f"Verification failed. Token received: {p.get('hub.verify_token')}")
    return Response(status_code=403)


@app.post("/webhook")
async def receive(request: Request, bg: BackgroundTasks):
    data = await request.json()
    bg.add_task(handle_message, data)
    return Response(status_code=200)


async def handle_message(data: dict):
    try:
        msgs = data["entry"][0]["changes"][0]["value"].get("messages", [])
        if not msgs:
            return
        msg         = msgs[0]
        from_number = msg["from"]
        msg_id      = msg["id"]

        if msg.get("type") != "text":
            await wa.send_text(from_number, "I can read text for now — type what you trained! 💪")
            return

        body = msg["text"]["body"].strip()
        print(f"Message from {from_number}: {body}")

        user  = store.get_or_create(from_number)
        await wa.send_reaction(from_number, msg_id, "💪")
        reply = await coach.handle_message(user, body)
        store.save(from_number, user)
        await wa.send_text(from_number, reply)

    except Exception as e:
        print(f"Handler error: {e}")

In [ ]:
# ── CELL 8: Start server + ngrok ─────────────────────────────────────────
import nest_asyncio
import uvicorn
import threading
import time
import sys
from pyngrok import ngrok, conf

nest_asyncio.apply()

# ── Step 1: Set ngrok auth token FIRST ───────────────────────────────────
if not NGROK_AUTH_TOKEN:
    raise ValueError(
        "NGROK_AUTH_TOKEN is missing!\n"
        "1. Sign up free at https://dashboard.ngrok.com/signup\n"
        "2. Copy your token from https://dashboard.ngrok.com/get-started/your-authtoken\n"
        "3. Add it as a Colab Secret named NGROK_AUTH_TOKEN\n"
        "4. Re-run Cell 2, then this cell"
    )

ngrok.kill()  # kill any stale tunnels from previous runs
conf.get_default().auth_token = NGROK_AUTH_TOKEN
print("✅ ngrok auth token set")

# ── Step 2: Start uvicorn in background thread ────────────────────────────
def run_server():
    sys.path.insert(0, '/content')  # ensure repsense package is found
    uvicorn.run(
        'repsense.app:app',
        host='0.0.0.0',
        port=8000,
        log_level='warning'
    )

server = threading.Thread(target=run_server, daemon=True)
server.start()
time.sleep(3)  # give uvicorn time to boot
print("✅ uvicorn server started on port 8000")

# ── Step 3: Open ngrok HTTPS tunnel ──────────────────────────────────────
tunnel     = ngrok.connect(8000, "http")
public_url = tunnel.public_url.replace("http://", "https://")
verify_tok = os.environ.get("WHATSAPP_VERIFY_TOKEN", "repsense_secret_2024")

print()
print("=" * 62)
print("🚀  RepSense AI is LIVE!")
print("=" * 62)
print()
print("📋  Paste these into Meta Dashboard → WhatsApp → Configuration:")
print()
print(f"   Callback URL :  {public_url}/webhook")
print(f"   Verify Token :  {verify_tok}")
print()
print("📝  Steps:")
print("   1. developers.facebook.com/apps → your app → WhatsApp → Configuration")
print("   2. Click Edit next to Webhook")
print("   3. Paste the Callback URL and Verify Token above")
print("   4. Click Verify and Save")
print("   5. Under Webhook Fields → enable: messages")
print()
print("💬  Then WhatsApp your test number and type: hello")
print("=" * 62)

In [ ]:
# ── CELL 9: Quick local test (verify Claude works before WhatsApp) ────────
import asyncio
import sys
sys.path.insert(0, '/content')

from repsense.ai_coach import AICoach
from repsense.store import UserStore

test_coach = AICoach()
test_store = UserStore()
user = test_store.get_or_create('test_local')

tests = [
    "hello",
    "just did squats 4x10 at 80kg",
    "/stats",
]

async def run():
    for msg in tests:
        print(f"\n👤 {msg}")
        reply = await test_coach.handle_message(user, msg)
        print(f"🤖 {reply}")
        print("-" * 50)

await run()
print("\n✅ Claude is responding — backend works!")

In [ ]:
# ── CELL 10: Keep-alive (run during demo to prevent Colab timeout) ────────
# Colab disconnects after ~90 min idle — this pings every 60s.
# Stop this cell when you're done.
import time
import httpx

print(f"🔄  Keeping server alive at {public_url}")
print("    Interrupt this cell (■) when done with your demo.")
print()

while True:
    try:
        r = httpx.get(public_url + "/", timeout=5)
        print(f"  ✅ [{time.strftime('%H:%M:%S')}] alive — HTTP {r.status_code}")
    except Exception as e:
        print(f"  ⚠️  ping failed: {e}")
    time.sleep(60)